7. Land multiple CSV files with slightly different formats (e.g., an extra column, a different delimiter) and document how your read_files() options need to change for each, plus how you'd detect a mismatch before it silently breaks downstream reports. 


In [0]:
-- File 1: Standard comma-delimited CSV
-- Options: format => 'csv', header => true, sep => ','
-- This is the simplest case — comma is the default delimiter.

SELECT * FROM read_files(
  '/Volumes/dev/default/csv_formats_demo/sales_comma.csv',
  format => 'csv',
  header => true,
  sep => ','
);

In [0]:
-- File 2: Pipe-delimited CSV
-- KEY CHANGE: sep => '|' instead of the default ','
-- If you forget this, every row becomes one giant column and the data is garbage.

SELECT * FROM read_files(
  '/Volumes/dev/default/csv_formats_demo/sales_pipe.csv',
  format => 'csv',
  header => true,
  sep => '|'
);

In [0]:
-- File 3: Tab-delimited CSV
-- KEY CHANGE: sep => '\t' (tab character)
-- Tabs are invisible in text viewers so this mismatch is easy to miss.

SELECT * FROM read_files(
  '/Volumes/dev/default/csv_formats_demo/sales_tab.csv',
  format => 'csv',
  header => true,
  sep => '\t'
);

In [0]:
-- File 4: Comma-delimited but with an EXTRA column (region)
-- The first 3 files have 4 columns; this file has 5.
-- Option 1: Let schema evolve — new column 'region' is auto-detected and added.

SELECT * FROM read_files(
  '/Volumes/dev/default/csv_formats_demo/sales_extra_col.csv',
  format => 'csv',
  header => true,
  sep => ',',
  schemaEvolutionMode => 'addNewColumns'
);

## Documentation: read_files() Options per Format & Mismatch Detection

### Option Changes per File Format

| File | Delimiter | Key Option Change | What Breaks if Wrong |
| --- | --- | --- | --- |
| `sales_comma.csv` | comma `,` | `sep => ','` (default) | Nothing — this is the baseline |
| `sales_pipe.csv` | pipe `\|` | `sep => '\|'` | Every row collapses into 1 column; all values land in `order_id` |
| `sales_tab.csv` | tab `\t` | `sep => '\t'` | Same as above — tabs are invisible so this is easy to miss |
| `sales_extra_col.csv` | comma `,` | `schemaEvolutionMode => 'addNewColumns'` or `'rescue'` | New `region` column is silently dropped or causes schema mismatch errors |

###Detect Mismatches Before They Break Downstream Reports

1. **Schema pre-check** (shown in Step 3): Read each file with `spark.read.csv()` using the expected delimiter, compare `df.columns` against the expected column list, and flag mismatches before loading into the target table.

2. **FAILFAST mode**: Use `.option("mode", "FAILFAST")` so Spark throws an error on the first malformed row instead of silently dropping or corrupting data.

3. **Rescue mode**: Set `schemaEvolutionMode => 'rescue'` so unexpected columns are captured in `_rescued_data` instead of being dropped — you can inspect and alert on non-null `_rescued_data` values.

4. **Column count / name assertions**: In a pipeline, add expectations like `expect("column_count", size(schema_fields) == 4)` to fail the pipeline if the schema changes.

5. **Monitoring**: Log `_rescued_data` row counts to a dashboard or alert so the data team is notified when a source file's format changes.

### Why This Matters

If a vendor adds a column or changes a delimiter and your pipeline doesn't detect it, downstream dashboards silently show nulls, wrong values, or truncated data. The mismatch detection step is the safety net that turns silent corruption into a visible alert.

8. Write a decision memo: when should Cyntexa choose Iceberg (or Delta UniForm) over native Delta for a given table, considering downstream tools like Snowflake or Trino? 

Naive delta - we will use the native delta when we use our mostly ecosystem in data bricks . cause data bricks have strong delta inegration and all delta feature are derectly available in data bricks.
also data bricks have ddefault native delta.

Ice berg - Cyntexa should consider Iceberg when the same data needs to be accessed by multiple data platforms such as Databricks, Trino, Snowflake, or other engines.
Iceberg is a good choice when open format interoperability is more important than Databricks-specific Delta features.

Delta Unifirm - Delta UniForm helps Cyntexa keep using Delta as the main table format while making the data accessible to engines that work with open table formats such as Iceberg.
So instead of maintaining separate Delta and Iceberg copies, Cyntexa can keep one underlying table and improve compatibility with external tools.
mean --> Use Delta UniForm when you want to keep Delta but also need better interoperability with external tools.


 9. Use DESCRIBE HISTORY together with the metadata columns to trace a specific bad row back to the exact ingestion run and source file that introduced it. 

In [0]:
select * from cyntexa_dev.default.customer_jsonread_metadata;

-- spouse the customer id 1004 is a bad record in table - cyntexa_dev.default.customer_jsonread_metadata.
-- table should contain the meta data 
-- from following query we can see the file name and file path the bad data inserted .

In [0]:

select * from cyntexa_dev.default.customer_jsonread_metadata where customer_id = 1004;

In [0]:
DESCRIBE HISTORY cyntexa_dev.default.customer_jsonread_metadata;